<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-05-llm-api-engineering/lesson-5.3-cost-engineering/notebooks/exercises-5.3.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 5.3 — Cost Engineering
Netsetos GenAI Course v2.0 | Module 5

Token optimization, prompt caching, batch API, tiered routing, cost observability, India planning


In [ ]:
# pip install tiktoken openai -q
import tiktoken
print('Ready for cost engineering')


## Ex 1: Token Counting & Cost Calculation


In [ ]:
enc = tiktoken.encoding_for_model('gpt-4o')

prompts = [
    'What is machine learning?',
    'Explain the transformer architecture in detail with examples',
    'You are an expert Python developer. Write a function to sort a list.',
]

models = {
    'GPT-4o': (2.50, 10.00),
    'GPT-4o-mini': (0.15, 0.60),
    'DeepSeek V3': (0.27, 1.10),
    'Gemini Flash': (0.075, 0.30),
}

for prompt in prompts:
    tokens = len(enc.encode(prompt))
    print(f'\nPrompt: "{prompt[:50]}..." → {tokens} tokens')
    for model, (inp, out) in models.items():
        cost = (tokens * inp / 1e6) + (200 * out / 1e6)  # assume 200 output
        print(f'  {model:15s}: ${cost:.6f} per request')


## Ex 2: System Prompt Optimization


In [ ]:
system_verbose = '''You are an expert customer service representative 
for TechCorp, a leading technology company. You have 
extensive knowledge of our products including smartphones, 
laptops, tablets, and smart home devices. You should 
always be polite, helpful, and professional. When a 
customer asks a question, provide detailed and accurate 
information. If you don't know the answer, say so 
honestly and offer to connect them with a specialist.'''

system_compressed = '''TechCorp support agent. Products: phones, laptops, 
tablets, smart home. Be helpful, accurate, concise. 
Unknown → offer specialist transfer.'''

enc = tiktoken.encoding_for_model('gpt-4o')
v_tokens = len(enc.encode(system_verbose))
c_tokens = len(enc.encode(system_compressed))

print(f'Verbose: {v_tokens} tokens')
print(f'Compressed: {c_tokens} tokens')
print(f'Reduction: {(1-c_tokens/v_tokens)*100:.0f}%')
print(f'\nAt 1M requests/month on GPT-4o:')
print(f'  Verbose: ${v_tokens * 1e6 / 1e6 * 2.50:.0f}/month')
print(f'  Compressed: ${c_tokens * 1e6 / 1e6 * 2.50:.0f}/month')
print(f'  Savings: ${(v_tokens - c_tokens) * 1e6 / 1e6 * 2.50:.0f}/month')


## Ex 3: Cost Stacking Calculator


In [ ]:
base_price = 3.00  # Claude Sonnet $/MTok input
monthly_tokens = 100  # 100M tokens

print('Cost Optimization Stacking:')
print(f'\nBaseline: {monthly_tokens}M tokens × ${base_price}/M = ${monthly_tokens * base_price:,.0f}/month')
print()

current = base_price
for name, mult, desc in [
    ('Prompt caching (90% off reads)', 0.10, '73% hit rate assumed'),
    ('Batch API (50% off)', 0.50, 'Non-real-time workloads'),
    ('Tiered routing (70% → cheap)', 0.30, '70% to $0.27/M models'),
]:
    current *= mult
    cost = monthly_tokens * current
    print(f'  + {name:40s} → ${cost:>8,.0f}/month ({mult}×)')

print(f'\nFinal: ${monthly_tokens * current:,.0f}/month')
print(f'Total reduction: {(1-current/base_price)*100:.1f}%')


## Ex 4: Indic Tokenization Penalty


In [ ]:
enc = tiktoken.encoding_for_model('gpt-4o')

texts = {
    'English': 'Machine learning is a subset of artificial intelligence that enables systems to learn from data.',
    'Hindi': 'मशीन लर्निंग आर्टिफिशियल इंटेलिजेंस का एक उपसमूह है जो सिस्टम को डेटा से सीखने में सक्षम बनाता है।',
}

print('Indic Tokenization Penalty:')
print()
for lang, text in texts.items():
    tokens = len(enc.encode(text))
    words = len(text.split())
    ratio = tokens / words
    print(f'{lang:10s}: {words} words → {tokens} tokens ({ratio:.2f} tokens/word)')

print()
print('Hindi costs ~63% more tokens than English on GPT-4o')
print('Tamil/Telugu: ~2× penalty. Llama 3.3: 4-5× for Hindi')
print('Mitigation: Sarvam AI tokenizer achieves ~1× for all Indic')


## Ex 5: Output Format Comparison


In [ ]:
enc = tiktoken.encoding_for_model('gpt-4o')

data_prose = '''The capital of India is New Delhi with a population of approximately 32 million people. The capital of Japan is Tokyo with a population of approximately 14 million people. The capital of France is Paris with a population of approximately 11 million people.'''

data_json = '{"capitals":[{"country":"India","capital":"New Delhi","pop_m":32},{"country":"Japan","capital":"Tokyo","pop_m":14},{"country":"France","capital":"Paris","pop_m":11}]}'

data_tsv = 'country\tcapital\tpop_m\nIndia\tNew Delhi\t32\nJapan\tTokyo\t14\nFrance\tParis\t11'

for name, text in [('Prose', data_prose), ('JSON', data_json), ('TSV', data_tsv)]:
    tokens = len(enc.encode(text))
    print(f'{name:6s}: {tokens} tokens ({len(text)} chars)')


## Ex 6: Budget Planning Calculator


In [ ]:
def budget_calc(model, input_price, output_price, 
                 monthly_requests, avg_input_tokens, avg_output_tokens,
                 cache_hit_rate=0, batch_pct=0, gst_pct=0):
    base_input = monthly_requests * avg_input_tokens / 1e6 * input_price
    base_output = monthly_requests * avg_output_tokens / 1e6 * output_price
    base = base_input + base_output
    
    cached = base_input * (1 - cache_hit_rate * 0.9)  # 90% cache discount
    batched = (cached + base_output) * (1 - batch_pct * 0.5)  # 50% batch
    gst = batched * (1 + gst_pct)
    
    print(f'{model}:')
    print(f'  Base API cost: ${base:,.0f}/month')
    print(f'  After caching ({cache_hit_rate*100:.0f}% hit): ${cached + base_output:,.0f}')
    print(f'  After batch ({batch_pct*100:.0f}%): ${batched:,.0f}')
    if gst_pct > 0:
        print(f'  After GST ({gst_pct*100:.0f}%): ${gst:,.0f} (₹{gst*93:,.0f})')
    print()

budget_calc('GPT-4o', 2.50, 10.00, 500_000, 800, 400)
budget_calc('GPT-4o (optimized)', 2.50, 10.00, 500_000, 800, 400,
            cache_hit_rate=0.7, batch_pct=0.3)
budget_calc('GPT-4o (India)', 2.50, 10.00, 500_000, 800, 400,
            cache_hit_rate=0.7, batch_pct=0.3, gst_pct=0.18)


## Ex 7: Embedding Model Cost Comparison


In [ ]:
models = [
    ('OpenAI text-embedding-3-small', 0.02, 62.3),
    ('OpenAI text-embedding-3-large', 0.13, 64.6),
    ('Cohere embed-v3', 0.10, 64.5),
    ('Gemini Embedding 001', 0.00, 63.8),  # free tier
    ('Self-hosted NV-Embed-v2', 0.001, 72.3),
]

docs = 1_000_000  # 1M documents
avg_tokens = 500  # per document
total_mtok = docs * avg_tokens / 1e6

print(f'Embedding 1M documents ({total_mtok:.0f}M tokens):')
print(f'{"Model":35s} {"$/MTok":>8s} {"Total $":>8s} {"MTEB":>6s}')
print('-'*62)
for name, price, mteb in sorted(models, key=lambda x: x[1]):
    cost = total_mtok * price
    print(f'{name:35s} ${price:>6.3f} ${cost:>7.1f} {mteb:>5.1f}')


## Ex 8: Cost-Per-Successful-Output


In [ ]:
print('Cost Paradox: CPSO (Cost Per Successful Output)')
print()
scenarios = [
    ('Claude Sonnet ($3/MTok)', 3.00, 0.95, 1500),
    ('Kimi K2.5 ($0.60/MTok)', 0.60, 0.40, 1500),
    ('GPT-4o-mini ($0.15/MTok)', 0.15, 0.85, 1500),
    ('DeepSeek V3 ($0.27/MTok)', 0.27, 0.70, 1500),
]

for model, price, success_rate, avg_tokens in scenarios:
    calls_needed = int(1000 / success_rate)
    total_tokens = calls_needed * avg_tokens / 1e6
    total_cost = total_tokens * price
    cpso = total_cost / 1000
    print(f'{model:30s}')
    print(f'  Success: {success_rate*100:.0f}% → {calls_needed} calls for 1000 outputs')
    print(f'  CPSO: ${cpso:.4f}/output  (Total: ${total_cost:.2f})')
    print()


---
## Recap
Tokens are only 30-40% of total production cost. Measure first (Helicone/Langfuse). Six optimization techniques compound to 50-80%: max_tokens, system prompt compression, few-shot reduction, stop sequences, output format (JSON 60% cheaper), embedding model selection (6-20× range). Cache (90% off) + batch (50% off) = 95% reduction. Tiered routing: 70% to budget models saves 60%+. Cost paradox: cheapest per-token ≠ cheapest per-task — measure CPSO. India: 18% GST + 63% Hindi token penalty. Sarvam AI eliminates Indic penalty with free LLMs. Budget enforcement: graduated thresholds at gateway layer.
